<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

## Appendix B: *WildFires*
- *Version Number: 4.0*
---
### Contents  
> *Wildfire Ignition Data*\
> *Wildfire Spread Data*\
> *Wildfire Damage Data*\
> *Combine Datasets*\
> *Export File*
---
### Notes
---
### Inputs
> - **Damage Data:**
>   - **CAL FIRE Damage Inspection (DINS)** 
>   - Data: <https://data.ca.gov/dataset/cal-fire-damage-inspection-dins-data>
> - **Ignition Data:**
>   - **Calfire Incident** 
>   - Data: <https://www.fire.ca.gov/incidents>
> - **Spread Data:**
>   - **The California Department of Forestry and Fire Protection's Fire and Resource Assessment Program (FRAP)** 
>   - Data: <https://data.ca.gov/dataset/california-fire-perimeters-all>
---
### Outputs  
- `fires_damage.csv`
- `fires_spread.csv`
- `fires_ignition.csv`
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# utility function that prints details of a datset
from src.data_utils import basic_explore

# utility function printing relevant details to check the health after a dataset merge
from src.data_utils import post_merge_check

# Function to map damages
from src.plot_utils import plot_fire_damage

### Third Party Dependencies  
---

In [2]:
# core data handling
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

## Ignition Data

### Load `CALfires.csv` - Wildfire Incident Data
Data obtained from [CAL FIRE Incident Data](https://www.fire.ca.gov/incidents)\
Dates of fire damage information: 01/01/2018 - 01/23/2025

> Variables Used:
> - `Fire name`
> - `Date started` and `Date Out` - Start and extinquishing date and time of wildfire incident 
> - `Total_Cost_Estimated` - in dollar amount
> - `County`, `Latitude` and `Longitude` - Spatial factors

Workflow:
- filter dataset by wildfire type
- column formatting
- fill NA values
- Ensure spatial accuracy of coordinates
- Format date and time columns

In [3]:
# Load data
allfires = pd.read_csv("../data/raw/Calfires.csv",low_memory = False)
allfires

,Unnamed: 0,SourceOID,ContainmentDateTime,IncidentSize,EstimatedCostToDate,FinalAcres,FireBehaviorGeneral,FireBehaviorGeneral1,FireBehaviorGeneral2,FireBehaviorGeneral3,...,LocalIncidentIdentifier,POOCity,POOCounty,POOState,PredominantFuelGroup,PredominantFuelModel,PrimaryFuelModel,SecondaryFuelModel,UniqueFireIdentifier,EstimatedFinalCost
0,0,7747595,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,066100,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2020-CALAC-066100,NaN
1,1,6384391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,009269,NaN,San Diego,US-CA,NaN,NaN,NaN,NaN,2019-CAMVU-009269,NaN
2,3,22499589,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,163915,NaN,Riverside,US-CA,NaN,NaN,NaN,NaN,2021-CARRU-163915,NaN
3,4,23869477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,396331,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2022-CALAC-396331,NaN
4,17,23247195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,008436,NaN,San Luis Obispo,US-CA,NaN,NaN,NaN,NaN,2020-CASLU-008436,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84534,364796,38438381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,242944,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2025-CALAC-242944,NaN
84535,364802,38438397,7/13/2025 4:02:20 PM,1.7,NaN,NaN,NaN,NaN,NaN,NaN,...,020730,NaN,Sacramento,US-CA,NaN,NaN,NaN,NaN,2025-CAAEU-020730,NaN
84536,364803,38438398,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,...,000813,NaN,Glenn,US-CA,NaN,NaN,NaN,NaN,2025-CAMNF-000813,NaN
84537,364819,38438435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,243075,NaN,Los Angeles,US-CA,NaN,NaN,NaN,NaN,2025-CALAC-243075,NaN


In [4]:
# Filter for wildfire incidents only
wildfires = allfires[allfires['IncidentTypeCategory']=='WF']

# rename for convenience
wildfires = wildfires.rename(columns={
    'InitialLatitude': 'Fire_Latitude',
    'InitialLongitude': 'Fire_Longitude',
    'FireDiscoveryDateTime':'Date',
    'POOCounty':'County'
})

# Drop entries with crucial null variables, may be manually fixed soon
wildfires = wildfires.dropna(subset=['Fire_Latitude', 'Fire_Longitude','Date'])


# Filter Coordinates
## Keep only coordinates that fall inside California boundaries since
## some entries have incorrect latitude and longitude values. 
## To be manually corrected in the future but are filtered out for now.

wildfires = wildfires[
    (wildfires['Fire_Latitude'] >= 32.5) & 
    (wildfires['Fire_Latitude'] <= 42) &
    (wildfires['Fire_Longitude'] >= -124.5) &
    (wildfires['Fire_Longitude'] <= -114)
]

# Format date and time columns
wildfires['Date'] = pd.to_datetime(wildfires['Date']).dt.date

# Fill Remaining NA values
## Entries with no cost present are assumed to have a damage cost 0 dollars.
wildfires['EstimatedFinalCost'] = wildfires['EstimatedFinalCost'].fillna(0)

drop =['Unnamed: 0', 'SourceOID', 'IncidentSize','FireBehaviorGeneral','EstimatedCostToDate','FireBehaviorGeneral1', 'FireBehaviorGeneral2', 'FireBehaviorGeneral3',
 'FireCause', 'FireCauseGeneral', 'FireCauseSpecific','IncidentShortDescription', 'IncidentTypeKind',
  'IrwinID','LocalIncidentIdentifier', 'POOCity', 'POOState','PredominantFuelGroup', 'PredominantFuelModel', 
  'PrimaryFuelModel','SecondaryFuelModel', 'UniqueFireIdentifier','IncidentTypeCategory','FireOutDateTime']

all_fires = wildfires.drop(columns=drop)

## Change Fire Name Field
all_fires['Fire Name'] = all_fires['IncidentName'].str.lower()
all_fires = all_fires.drop(columns='IncidentName')

In [ ]:
all_fires['End'] = all_fires['ContainmentDateTime'].fillna(all_fires['Date'])

all_fires['End'] = pd.to_datetime(all_fires['End'])
all_fires['Date'] = pd.to_datetime(all_fires['Date'])

all_fires['Days Burned'] = (all_fires['End'] - all_fires['Date']).dt.total_seconds() / (24*3600)

all_fires['Days Burned'] += 1

all_fires


In [ ]:
damaging_fires = all_fires[all_fires['EstimatedFinalCost']>0]

mean_burntime = int(
    np.ceil(damaging_fires['Days Burned'].mean())
)

mean_burntime

## Fire Spread

In [14]:
perimeters = pd.read_csv("../data/raw/California_Fire_Perimeters.csv",low_memory = False)

In [15]:
keep = ['FIRE_NAME', 'ALARM_DATE', 'CONT_DATE', 'GIS_ACRES', 'Centroid_x',
       'Centroid_y']

perimeters = perimeters[keep]

In [16]:
perimeters = perimeters.rename(columns={
    'ALARM_DATE': 'Date',
    'FIRE_NAME': 'Fire_Name',
    'GIS_ACRES':'Acres',
    'CONT_DATE': 'Fire_Contained'
})

In [17]:
perimeters

,Fire_Name,Date,Fire_Contained,Acres,Centroid_x,Centroid_y
0,PALISADES,1/7/2025 0:00:00,1/31/2025 0:00:00,23448.900391,131594.509709,-437120.204741
1,EATON,1/8/2025 0:00:00,1/31/2025 0:00:00,14056.299805,175755.502049,-422125.449066
2,HUGHES,1/22/2025 0:00:00,1/28/2025 0:00:00,10396.799805,129867.003200,-385044.138995
3,KENNETH,1/9/2025 0:00:00,2/4/2025 0:00:00,998.737976,121277.591045,-425898.009312
4,HURST,1/7/2025 0:00:00,1/9/2025 0:00:00,831.385010,139844.001565,-408130.444855
...,...,...,...,...,...,...
2757,CANADA,9/18/2018 0:00:00,9/18/2018 0:00:00,0.003500,68717.065635,-406215.845103
2758,ERRINGER,1/31/2018 0:00:00,1/31/2018 0:00:00,0.002821,115448.082864,-414059.216904
2759,NO NAME,7/30/2018 0:00:00,7/30/2018 0:00:00,0.002527,68003.729531,-397177.288343
2760,HARBOR,1/29/2018 0:00:00,1/29/2018 0:00:00,0.001796,63753.994868,-415195.625835


In [ ]:
perimeters['Fire_Contained'] = pd.to_datetime(perimeters['Fire_Contained'])
perimeters['Date'] = pd.to_datetime(perimeters['Date'])

perimeters.loc[perimeters['Fire_Contained'] < perimeters['Date'],'Fire_Contained'] = perimeters['Date']

perimeters['Days Burned'] = (perimeters['Fire_Contained'] - perimeters['Date']).dt.total_seconds() / (24*3600)
perimeters['Days Burned'] += 1
perimeters['Days Burned']

In [ ]:
perimeters['Acres per Day'] = perimeters['Acres'] / perimeters['Days Burned']
perimeters['Acres per Day']

In [24]:
perimeters_expanded = (
    perimeters
    .loc[perimeters.index.repeat(perimeters['Days Burned'])]
    .assign(
        Day_Offset=lambda x: x.groupby(level=0).cumcount(),
        Date=lambda x: x['Date'] + pd.to_timedelta(x['Day_Offset'], unit='D'),
        Acres_Burned_So_Far=lambda x: x['Acres per Day'] * (x['Day_Offset'] + 1)
    )
    .drop(columns='Day_Offset')
    .reset_index(drop=True)
)

In [25]:
perimeters_expanded

,Fire_Name,Date,Fire_Contained,Acres,Centroid_x,Centroid_y,Days Burned,Acres per Day,Acres_Burned_So_Far
0,PALISADES,2025-01-07,2025-01-31,23448.900391,131594.509709,-437120.204741,25.0,937.956016,937.956016
1,PALISADES,2025-01-08,2025-01-31,23448.900391,131594.509709,-437120.204741,25.0,937.956016,1875.912031
2,PALISADES,2025-01-09,2025-01-31,23448.900391,131594.509709,-437120.204741,25.0,937.956016,2813.868047
3,PALISADES,2025-01-10,2025-01-31,23448.900391,131594.509709,-437120.204741,25.0,937.956016,3751.824062
4,PALISADES,2025-01-11,2025-01-31,23448.900391,131594.509709,-437120.204741,25.0,937.956016,4689.780078
...,...,...,...,...,...,...,...,...,...
25907,CANADA,2018-09-18,2018-09-18,0.003500,68717.065635,-406215.845103,1.0,0.003500,0.003500
25908,ERRINGER,2018-01-31,2018-01-31,0.002821,115448.082864,-414059.216904,1.0,0.002821,0.002821
25909,NO NAME,2018-07-30,2018-07-30,0.002527,68003.729531,-397177.288343,1.0,0.002527,0.002527
25910,HARBOR,2018-01-29,2018-01-29,0.001796,63753.994868,-415195.625835,1.0,0.001796,0.001796


## Fire Damage Processing

`Damage_data_master` - Wildfire Incident Data on Structural Damage

Data obtained from CALFIRE website. Includes data on the degree of damages for every location since 1950. 

- Fire name and date started
- Damage levels (structures destroyed/damaged)
- County, Latitude, Longitude of event

In [26]:
raw_damages = pd.read_csv("../data/raw/Damage_data_master.csv",low_memory = False)
basic_explore(raw_damages)

Rows:  130722  Columns:  46
Duplicates  0
Total NA values:  1381634  of  6013212 datapoints


In [27]:
raw_damages.columns

Index(['OBJECTID', '* Damage', '* Street Number', '* Street Name',
       '* Street Type (e.g. road, drive, lane, etc.)',
       'Street Suffix (e.g. apt. 23, blding C)', '* City', 'State', 'Zip Code',
       '* CAL FIRE Unit', 'County', 'Community', 'Battalion',
       '* Incident Name', 'Incident Number (e.g. CAAEU 123456)',
       'Incident Start Date', 'Hazard Type',
       'If Affected 1-9% - Where did fire start?',
       'If Affected 1-9% - What started fire?',
       'Structure Defense Actions Taken', '* Structure Type',
       'Structure Category', '# Units in Structure (if multi unit)',
       '# of Damaged Outbuildings < 120 SQFT',
       '# of Non Damaged Outbuildings < 120 SQFT', '* Roof Construction',
       '* Eaves', '* Vent Screen', '* Exterior Siding', '* Window Pane',
       '* Deck/Porch On Grade', '* Deck/Porch Elevated',
       '* Patio Cover/Carport Attached to Structure',
       '* Fence Attached to Structure', 'Distance - Propane Tank to Structure',
       'Dis

### Preprocess Fire Damage Records

- Filtered down to essential columns: fire name, type, category, damage, date, and county.
- Renamed columns for consistency.
- Retained only fire-related records (excluding other hazard types).
- Convert the start date to `datetime.date` and rename.

In [28]:
# filter to columns to keep
old_col_names = ['* Damage','County','Incident Start Date',
                 'Hazard Type', 'Structure Category','* Structure Type','* Incident Name',
                'Latitude','Longitude']

damages = raw_damages[old_col_names]

# rename for convenience
damages.columns = ['Damage','County', 'Start','Hazard','Category',
                  'Type','Fire Name','Lat','Lon']

# Filter for only fires, drop Hazard category, it is no longer needed
damages = damages[damages['Hazard'] == 'Fire']
damages = damages.drop(columns = ['Hazard'])

# filter the dataset to current operating time range
damages['Date'] = pd.to_datetime(damages['Start'], format='mixed').dt.date
damages = damages.drop(columns = ['Start'])

In [29]:
basic_explore(damages)

Rows:  130720  Columns:  8
Duplicates  108
Total NA values:  30  of  1045760 datapoints


In [30]:
damages.isna().sum()

Damage        0
County       30
Category      0
Type          0
Fire Name     0
Lat           0
Lon           0
Date          0
dtype: int64

**Note:**  
Approximately 30 records were missing county names in the original data. However, ZIP code and address information allowed inference of the correct county.  
Due to the small number, these corrections were applied manually by index.  
- Rows `78763–78773`: Riverside County  
- Rows `78435–78451`: Yuba County  
- Rows `88665` and `88681`: Tulare County  

These updates ensure completeness for spatial grouping and aggregation in downstream analysis.

In [31]:
riverside_ids = range(78763,78774)
yuba_ids = range(78435,78452)
damages.loc[riverside_ids,'County'] = 'Riverside'
damages.loc[yuba_ids,'County'] = 'Yuba'
damages.loc[88665,'County'] = 'Tulare'
damages.loc[88681,'County'] = 'Tulare'

In [32]:
damages.head(2)

,Damage,County,Category,Type,Fire Name,Lat,Lon,Date
0,No Damage,Solano,Single Residence,Single Family Residence Multi Story,Quail,38.474960,-122.044465,2020-06-06
1,Affected (1-9%),Solano,Single Residence,Single Family Residence Single Story,Quail,38.477442,-122.043252,2020-06-06


### Calculate Damages

#### Estimate Dollar Value of Structure Damage

To approximate the economic cost of fire damage, a base dollar value was assigned to each general structure type:

In [33]:
base_value_map = {
    'Single Residence': 800000,
    'Multiple Residence': 1000000,
    'Mixed Commercial/Residential': 1500000,
    'Nonresidential Commercial': 2000000,
    'Other Minor Structure': 10000,
    'Infrastructure': 1000000,
    'Agriculture': 10000,
}

#### Value Modifiers for Building Descriptors

The estimated dollar value for each damaged structure is adjusted based on detailed descriptors. These modifiers scale the base value assigned to the general structure type.

> **Note:** These modifiers are estimates that reflect relative structural value and are not based on official costs.

In [34]:
type_modifier_map = {
    'Single Family Residence Multi Story': 1.0,
    'Single Family Residence Single Story': 0.95,
    'Utility Misc Structure': 0.1,
    'Mobile Home Double Wide': 0.5,
    'Motor Home': 0.3,
    'Multi Family Residence Multi Story': 1.2,
    'Commercial Building Single Story': 1.0,
    'Mobile Home Single Wide': 0.4,
    'Mixed Commercial/Residential': 1.5,
    'Mobile Home Triple Wide': 0.6,
    'Infrastructure': 0.8,
    'School': 1.3,
    'Multi Family Residence Single Story': 1.1,
    'Commercial Building Multi Story': 1.4,
    'Church': 1.0,
    'Hospital': 1.6,
    'Agriculture': 0.2,
    'Single Famliy Residence Single Story': 0.95,
    'Utility or Miscellaneous Structure > 120 sqft':0.1
}

#### Lookup Table: Estimated Damage Values by Structure Type and Descriptor

The estimated monetary damage per structure is computed by multiplying:
- A base value associated with the general **structure category** (Single Residence)
- A modifier associated with the **building descriptor** (Single Story, Mobile Home)

This matrix provides a quick reference for assigning dollar estimates to individual records based on their structure type and category.

> **Formula:**  
> `Estimated Value = Base Value × Type Modifier`

In [35]:
# Create a DataFrame from the product of categories and types
value_matrix = pd.DataFrame(
    [(cat, typ, base_value_map[cat] * type_modifier_map[typ]) for cat in base_value_map for typ in type_modifier_map],
    columns=['Category', 'Type', 'Adjusted Value']
)

# Pivot the DataFrame to get a matrix format
value_matrix_df = value_matrix.pivot(index='Category', columns='Type', values='Adjusted Value')

# Display the resulting matrix
value_matrix_df

Type,Agriculture,Church,Commercial Building Multi Story,Commercial Building Single Story,Hospital,Infrastructure,Mixed Commercial/Residential,Mobile Home Double Wide,Mobile Home Single Wide,Mobile Home Triple Wide,Motor Home,Multi Family Residence Multi Story,Multi Family Residence Single Story,School,Single Family Residence Multi Story,Single Family Residence Single Story,Single Famliy Residence Single Story,Utility Misc Structure,Utility or Miscellaneous Structure > 120 sqft
Category,,,,,,,,,,,,,,,,,,,
Agriculture,2000.0,10000.0,14000.0,10000.0,16000.0,8000.0,15000.0,5000.0,4000.0,6000.0,3000.0,12000.0,11000.0,13000.0,10000.0,9500.0,9500.0,1000.0,1000.0
Infrastructure,200000.0,1000000.0,1400000.0,1000000.0,1600000.0,800000.0,1500000.0,500000.0,400000.0,600000.0,300000.0,1200000.0,1100000.0,1300000.0,1000000.0,950000.0,950000.0,100000.0,100000.0
Mixed Commercial/Residential,300000.0,1500000.0,2100000.0,1500000.0,2400000.0,1200000.0,2250000.0,750000.0,600000.0,900000.0,450000.0,1800000.0,1650000.0,1950000.0,1500000.0,1425000.0,1425000.0,150000.0,150000.0
Multiple Residence,200000.0,1000000.0,1400000.0,1000000.0,1600000.0,800000.0,1500000.0,500000.0,400000.0,600000.0,300000.0,1200000.0,1100000.0,1300000.0,1000000.0,950000.0,950000.0,100000.0,100000.0
Nonresidential Commercial,400000.0,2000000.0,2800000.0,2000000.0,3200000.0,1600000.0,3000000.0,1000000.0,800000.0,1200000.0,600000.0,2400000.0,2200000.0,2600000.0,2000000.0,1900000.0,1900000.0,200000.0,200000.0
Other Minor Structure,2000.0,10000.0,14000.0,10000.0,16000.0,8000.0,15000.0,5000.0,4000.0,6000.0,3000.0,12000.0,11000.0,13000.0,10000.0,9500.0,9500.0,1000.0,1000.0
Single Residence,160000.0,800000.0,1120000.0,800000.0,1280000.0,640000.0,1200000.0,400000.0,320000.0,480000.0,240000.0,960000.0,880000.0,1040000.0,800000.0,760000.0,760000.0,80000.0,80000.0


#### Assign Estimated Dollar Values to Each Record

Using the lookup matrix, each damage record is assigned an estimated economic value by matching its structure `Category` and `Type`.

- The assigned value represents an approximate replacement or reconstruction cost.
- These values are based on estimates and are used to generate a composite damage index for modeling purposes.

Validation:
- Any Rows with unmatched (Category, Type) combinations are flagged (`NaN`).


In [36]:
damages['Adjusted Value'] = damages.apply(
    lambda row: value_matrix_df.loc[row['Category'], row['Type']], axis=1
)

print('Adjusted Values NA values : ', damages['Adjusted Value'].isna().sum())
total = damages['Adjusted Value'].values.sum()
print(f"Total Adjusted Value: ${total:,.0f}")

Adjusted Values NA values :  0
Total Adjusted Value: $77,939,291,500


In [37]:
damage_weights = {
    'Destroyed (>50%)': 1.0,
    'Major (26-50%)': 0.4,
    'Minor (10-25%)': 0.2,
    'Affected (1-9%)': 0.05,
    'Inaccessible': 0.3,
    'No Damage': 0.0
}

In [38]:
# Assume each row is one structure. Map category to loss fraction:
damages['Loss Fraction'] = damages['Damage'].map(damage_weights)

# Calculate estimated damage per structure
damages['Estimated Damage'] = damages['Loss Fraction'] * damages['Adjusted Value']

#### Sanity Checks

In [39]:
damages.isna().sum()

Damage              0
County              0
Category            0
Type                0
Fire Name           0
Lat                 0
Lon                 0
Date                0
Adjusted Value      0
Loss Fraction       0
Estimated Damage    0
dtype: int64

In [40]:
damages['Damage'].value_counts()

Damage
Destroyed (>50%)    70127
No Damage           53051
Affected (1-9%)      5023
Minor (10-25%)       1337
Major (26-50%)        706
Inaccessible          476
Name: count, dtype: int64

#### Aggregate Damage Estimates by Fire Event

Now that each damage record has an estimated dollar value, we summarize the data:

- Group by **Fire Name** and **Fire Start Date**.
- Sum the `Adjusted Value` to get total `Estimated Damage` per fire event.

In [41]:
fire_damages = damages.groupby(['Fire Name', 'Date']).agg({
    'Estimated Damage': 'sum',
    'Lat': 'first',
    'Lon': 'first',
    'County': 'first'
}).reset_index()

In [42]:
fire_damages.rename(columns={'Lat':'Fire_Latitude','Lon':'Fire_Longitude'}, inplace = True)

In [43]:
fire_damages.isna().sum()

Fire Name           0
Date                0
Estimated Damage    0
Fire_Latitude       0
Fire_Longitude      0
County              0
dtype: int64

In [44]:
## sort dataframe by date
fire_damages = fire_damages.sort_values(by='Date')

## reset index
fire_damages = fire_damages.reset_index(drop=True)

In [45]:
 positive_fire_damages = fire_damages[fire_damages['Estimated Damage'] > 0]

In [46]:
 positive_fire_damages['Damage per Day'] =  positive_fire_damages['Estimated Damage'] / mean_burntime

C:\Users\dusti\AppData\Local\Temp\ipykernel_23184\725370427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_fire_damages['Damage per Day'] =  positive_fire_damages['Estimated Damage'] / mean_burntime


In [47]:
fire_damages_expanded = (
    positive_fire_damages
    .loc[positive_fire_damages.index.repeat(mean_burntime)]
    .assign(
        Day_Offset=lambda x: x.groupby(level=0).cumcount(),
        Date=lambda x: x['Date'] + pd.to_timedelta(x['Day_Offset'], unit='D'),
        Damage_So_Far=lambda x: x['Damage per Day'] * (x['Day_Offset'] + 1)
    )
    .drop(columns='Day_Offset')
    .reset_index(drop=True)
)

C:\Users\dusti\AppData\Local\Temp\ipykernel_23184\3776325513.py:6: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  Date=lambda x: x['Date'] + pd.to_timedelta(x['Day_Offset'], unit='D'),


In [48]:
fire_damages_expanded

,Fire Name,Date,Estimated Damage,Fire_Latitude,Fire_Longitude,County,Damage per Day,Damage_So_Far
0,Silver,2013-08-07,2.114200e+07,33.871821,-116.832461,Riverside,6.218235e+05,6.218235e+05
1,Silver,2013-08-08,2.114200e+07,33.871821,-116.832461,Riverside,6.218235e+05,1.243647e+06
2,Silver,2013-08-09,2.114200e+07,33.871821,-116.832461,Riverside,6.218235e+05,1.865471e+06
3,Silver,2013-08-10,2.114200e+07,33.871821,-116.832461,Riverside,6.218235e+05,2.487294e+06
4,Silver,2013-08-11,2.114200e+07,33.871821,-116.832461,Riverside,6.218235e+05,3.109118e+06
...,...,...,...,...,...,...,...,...
9719,Palisades,2025-02-05,4.698816e+09,34.042652,-118.618492,Los Angeles,1.382005e+08,4.146014e+09
9720,Palisades,2025-02-06,4.698816e+09,34.042652,-118.618492,Los Angeles,1.382005e+08,4.284215e+09
9721,Palisades,2025-02-07,4.698816e+09,34.042652,-118.618492,Los Angeles,1.382005e+08,4.422415e+09
9722,Palisades,2025-02-08,4.698816e+09,34.042652,-118.618492,Los Angeles,1.382005e+08,4.560616e+09


## Export File

In [49]:
fire_damages_expanded.to_csv("../data/processed/fires_damage.csv",index=False)
perimeters_expanded.to_csv('../data/processed/fires_spread.csv',index = False)
all_fires.to_csv('../data/processed/fires_ignition.csv', index=False)
print("All datasets saved successfully.")

All datasets saved successfully.
